In [ ]:
import pandas as pd
from sklearn.metrics import root_mean_squared_log_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance


projects = pd.read_csv("train.csv")

In [ ]:
projects.columns

In [ ]:
y = projects.hours_spent
# Numerical features
project_features = ['num_commits', 'active_commit_days', 'project_span_days',
       'pct_commits_weekend', 'pct_commits_late_night', 'lines_added_total',
       'lines_deleted_total', 'churn_ratio', 'median_lines_per_commit',
       'max_lines_in_one_commit', 'pct_code_in_first_commit',
       'num_commit_authors', 'avg_commit_msg_length',
       'pct_generic_commit_msgs', 'num_branches', 'num_merge_commits',
       'lines_of_code', 'num_source_files', 'num_directories', 'max_dir_depth',
       'num_languages', 'pct_primary_language', 'num_dependencies',
       'num_test_files', 'avg_file_length', 'comment_line_ratio',
       'num_image_files', 'num_audio_files', 'num_3d_model_files',
       'num_font_files', 'asset_size_mb', 'readme_word_count',
       'readme_num_headings', 'readme_num_images', 'readme_num_code_blocks',
       'page_weight_kb', 'num_js_files_loaded', 'num_internal_pages',
       'load_time_ms', 'description_word_count', 'stars', 'forks',
       'num_issues', 'num_pull_requests', 'num_releases',
       'has_custom_art', 'is_mobile_responsive', 'is_multiplayer', 
       'mentions_ai_api', 'is_clone_or_tutorial']

print(len(project_features))

object_cols = ['has_custom_art', 'is_mobile_responsive', 
               'is_multiplayer', 'mentions_ai_api', 
               'is_clone_or_tutorial']

drop = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
X = projects.drop(columns=drop)

cat_cols = X.select_dtypes(include=["str", "object"]).columns
X[cat_cols] = X[cat_cols].astype("category")

X.head()

In [ ]:
project_model = HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0)

# train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)
# label_train_X = train_X.copy()
# label_val_X = val_X.copy()

# OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(train_X[object_cols]))
# OH_cols_valid = pd.DataFrame(OH_encoder.transform(val_X[object_cols]))

# OH_cols_train.index = train_X.index
# OH_cols_valid.index = val_X.index

# num_X_train = train_X.drop(object_cols, axis=1)
# num_X_valid = val_X.drop(object_cols, axis=1)

# OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
# OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# OH_X_train.columns = OH_X_train.columns.astype(str)
# OH_X_valid.columns = OH_X_valid.columns.astype(str)

# project_model.fit(OH_X_train, np.log1p(train_y))

# print("Predictions for test projects")
# print(project_model.predict(OH_X_valid))
# print("True log hours")
# print(np.log1p(val_y))

# root_mean_squared_log_error(val_y, np.expm1(project_model.predict(OH_X_valid)))
print("RMSLE")
scores = cross_val_score(project_model, X, np.log1p(y), cv=5,
                         scoring="neg_root_mean_squared_error")
print(-scores.mean(), scores.std())

In [ ]:
# Replacing cus i think dividning by zero is causing an error
X['commits_per_day'] = X['num_commits'] / X['active_commit_days'].replace(0, np.nan)
X['total_changed'] = X['lines_added_total'] + X['lines_deleted_total']
X['active_day_ratio'] = X['active_commit_days'] / X['project_span_days'].replace(0, np.nan)
X['added_lines_per_day'] = X['lines_added_total'] / X['active_commit_days'].replace(0, np.nan)
X['removed_lines_per_day'] = X['lines_deleted_total'] / X['active_commit_days'].replace(0, np.nan)
X['changed_lines_per_day'] = X['total_changed'] / X['active_commit_days'].replace(0, np.nan)

In [ ]:
# Fine tuning the params for best score

project_model = HistGradientBoostingRegressor(
    categorical_features="from_dtype", 
    learning_rate=0.03, max_iter=10000,
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=100, random_state=0)
scores = cross_val_score(project_model, X, np.log1p(y), cv=5,
                        scoring="neg_root_mean_squared_error")
print(-scores.mean(), scores.std())

In [ ]:
project_model.predict([])

In [ ]:
project_model.fit(X, np.log1p(y))
r = permutation_importance(project_model, X, np.log1p(y), n_repeats=3, random_state=0)


In [ ]:
pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False).head(15)

In [ ]:
# Me testing some guesses manually
row = X.iloc[[0]].copy()
row['project_category'] = "app"
row['primary_language'] = 'HTML'
row['num_commits'] = 21
row['active_commit_days'] = 6
row['lines_of_code'] = 1100
row['num_source_files'] = 6
row['readme_word_count'] = 118
row["url_is_live"] = False

row['commits_per_day'] = row['num_commits'] / row['active_commit_days'].replace(0, np.nan)

log_prediction = project_model.predict(row)
hours = np.expm1(log_prediction)
print(hours)
